Download [dataset](https://www.kaggle.com/competitions/classification-of-butterflies)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import random

seed = 42
random.seed(seed)
torch.random.manual_seed(seed)
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
learning_rate = 1e-3
batch_size = 64
epochs = 1000
sample_rate = 0.2

device

In [ ]:
import os
import cv2
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, random_split
from samplefier import Samplefier

samplefier = Samplefier(seed, sample_rate)

def normalize(images: np.ndarray) -> torch.Tensor:  # SHWC -> SCHW
    images = np.transpose(images, (0, 3, 1, 2)).astype(np.float32)
    mean = np.mean(images, axis=(1, 2, 3), keepdims=True, dtype=np.float32)
    std = np.std(images, axis=(1, 2, 3), keepdims=True, dtype=np.float32)
    std = np.clip(std, 1e-6, None)
    return ((images - mean) / std).astype(np.float32)

class TrainButterflyDataset(Dataset):
    def __init__(self):
        self.data = []
        self.labels = []
        for folder in tqdm(os.listdir("data/train_split/"), desc="Loading train classes"):
            class_name = int(folder[folder.rfind("_") + 1:])
            for file in os.listdir(f"data/train_split/{folder}"):
                image = cv2.imread(f"data/train_split/{folder}/{file}")
                np_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                self.data.append(np_image)
                self.labels.append(class_name)
                if random.random() < sample_rate:
                    sampled_images = samplefier(np_image)
                    self.data.extend(sampled_images)
                    self.labels.extend([class_name] * len(sampled_images))

        print("Normalizing data")
        self.data = normalize(np.array(self.data, dtype=np.float32))
        self.labels = np.array(self.labels, dtype=np.int64)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class TestButterflyDataset(Dataset):
    def __init__(self):
        self.data = []
        for file in tqdm(os.listdir("data/valid/"), desc="Loading test images"):
            image = cv2.imread(f"data/valid/{file}")
            np_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32)
            np_image = np.transpose(np_image, (2, 0, 1))  # HWC -> CHW
            self.data.append(np_image)
        self.data = np.array(self.data, dtype=np.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

full_train_dataset = TrainButterflyDataset()
train_dataset, valid_dataset = random_split(full_train_dataset, [0.8, 0.2])
test_dataset = TestButterflyDataset()

full_train_loader = DataLoader(full_train_dataset, batch_size=batch_size, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print('-'*100)
print(f"Train length: {len(train_dataset)}")
print(f"Valid length: {len(valid_dataset)}")
print(f"Test length: {len(test_dataset)}")


In [ ]:
full_train_dataset[0][0].shape

In [ ]:
import torch.nn as nn
import torch.optim as optim

def conv_block(in_c, out_c):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(),
        nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )

class ButterflyNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 256),
            conv_block(256, 512),
        )
        self.head = nn.Sequential(
            self.features,
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.head(x)


num_classes = int(full_train_dataset.labels.max()) + 1
net = ButterflyNet(num_classes).to(device)
optimizer = optim.Adam(net.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss(reduction="mean")
scheduler = lambda opt: optim.lr_scheduler.ReduceLROnPlateau(opt, "min", patience=10, factor=0.1)

In [ ]:
import os
import sys
import importlib
from IPython.display import display
import matplotlib.pyplot as plt

# Ensure we import the correct trainer.py for this homework folder.
# Also reload the module because Jupyter may cache the old version.
trainer_dir = (
    os.path.abspath("Homework_13_butterfly_classification")
    if os.path.isdir("Homework_13_butterfly_classification")
    else os.path.abspath(".")
)
sys.path.insert(0, trainer_dir)

# Force re-import from the desired path (avoids stale cached module)
if "trainer" in sys.modules:
    del sys.modules["trainer"]

import trainer as trainer_module
trainer_module = importlib.reload(trainer_module)
Trainer = trainer_module.Trainer

# Live loss plot (updates after each epoch)
fig, ax = plt.subplots(figsize=(7, 4))
train_line, = ax.plot([], [], label="train")
val_line, = ax.plot([], [], label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.grid(True, alpha=0.3)
ax.legend()
handle = display(fig, display_id=True)

epoch_numbers: list[int] = []
train_losses: list[float] = []
val_losses: list[float] = []

def on_epoch_end(epoch: int, train_loss: float, val_loss: float | None) -> None:
    epoch_numbers.append(epoch + 1)
    train_losses.append(train_loss)
    if val_loss is not None:
        val_losses.append(val_loss)

    train_line.set_data(epoch_numbers, train_losses)
    if val_losses:
        val_line.set_data(epoch_numbers[: len(val_losses)], val_losses)

    ax.relim()
    ax.autoscale_view()
    handle.update(fig)

trainer = Trainer(
    net,
    criterion,
    optimizer,
    device,
    epoch_amount=epochs,
    scheduler=scheduler,
    epoch_end_callback=on_epoch_end,
)
trainer.fit(train_loader, valid_loader)

In [ ]:
torch.save(trainer.best_model.state_dict(), "data/butterfly_net.pt")

In [ ]:
pred = trainer.predict(test_loader)
classes = pred.softmax(dim=1).argmax(dim=1).numpy()
pd.DataFrame({"index": np.arange(0, len(classes)), "label": classes}).to_csv(
    "data/predictions.csv", index=False
)
